# 04 CNN Dropout Regularisation Trial

Controlled hyperparameter screening. Change only `dropout`; all inherited settings remain fixed. No L2 is added unless a later human decision changes the methodology.


## 1. Package Setup


In [1]:
# Purpose: Package installs are documented but not run automatically during this static refactor.
# %pip install tensorflow scikit-learn pandas numpy matplotlib seaborn joblib soundfile librosa


## 2. Load Cache and Previous Config


In [2]:
import os
from pathlib import Path

# Purpose: Mounts Google Drive when this notebook is running in Google Colab.
# Why this exists: the project files, cached MFCC features, manifests, models, figures,
# and metric outputs live in Google Drive during Colab runs. The /content/drive path
# only represents the real MyDrive files after drive.mount("/content/drive") succeeds.
# Important: the project root should be the folder that contains Data, Model Variants,
# and outputs. For this project, that expected Colab folder is the INM701 folder below.
COLAB_DRIVE_MOUNT_POINT = Path("/content/drive")
EXPECTED_COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")

try:
    from google.colab import drive

    drive.mount(str(COLAB_DRIVE_MOUNT_POINT))
    print("Google Colab detected. Google Drive mounted.")

    current_project_root = os.environ.get("INTRO_AI_PROJECT_ROOT")
    current_data_dir_exists = bool(current_project_root) and (Path(current_project_root) / "Data").exists()
    expected_data_dir_exists = (EXPECTED_COLAB_PROJECT_ROOT / "Data").exists()

    # Purpose: Keep a valid user-provided project root, but repair stale runtime state
    # if a previous cell pointed INTRO_AI_PROJECT_ROOT somewhere that does not contain Data.
    if current_data_dir_exists:
        print("INTRO_AI_PROJECT_ROOT already points to a folder with Data, so it was kept.")
    elif expected_data_dir_exists:
        os.environ["INTRO_AI_PROJECT_ROOT"] = str(EXPECTED_COLAB_PROJECT_ROOT)
        print("INTRO_AI_PROJECT_ROOT set to the expected INM701 project folder.")
    elif not current_project_root:
        os.environ["INTRO_AI_PROJECT_ROOT"] = str(EXPECTED_COLAB_PROJECT_ROOT)
        print("INTRO_AI_PROJECT_ROOT was not set, so it now points to the expected INM701 folder.")
    else:
        print("INTRO_AI_PROJECT_ROOT was kept, but Data was not found there or in the expected INM701 folder.")
except Exception as exc:
    print("Google Colab Drive mount skipped. This is expected outside Colab.")
    print("Mount skip reason:", exc)

active_project_root = os.environ.get("INTRO_AI_PROJECT_ROOT", "not set")
print("INTRO_AI_PROJECT_ROOT:", active_project_root)
if active_project_root != "not set":
    active_project_root = Path(active_project_root)
    print("Project root exists:", active_project_root.exists())
    print("Expected Data folder:", active_project_root / "Data")
    print("Data folder exists:", (active_project_root / "Data").exists())


Mounted at /content/drive
Google Colab detected. Google Drive mounted.
INTRO_AI_PROJECT_ROOT set to the expected INM701 project folder.
INTRO_AI_PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Education/INM701
Project root exists: True
Expected Data folder: /content/drive/MyDrive/Colab Notebooks/Education/INM701/Data
Data folder exists: True


In [3]:
# Purpose: Later notebooks load the one CNN-ready cache made by notebook 00. They do not
# rescan folders, regenerate splits, extract MFCCs, fit scalers, or recalculate class weights.
import hashlib
import json
import os
import random
import time
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from tensorflow.keras.layers import Conv2D, Dense, Dropout, GlobalAveragePooling2D, Input, MaxPooling2D
from tensorflow.keras.models import Sequential

try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 7
EARLY_STOP_MIN_DELTA = 1e-4
THRESHOLD = 0.5
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}
CONFIRMATION_SEEDS = [42, 123, 2026]


def resolve_project_root():
    # Purpose: default path.
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd().resolve()
    # Purpose: Runs each candidate configuration under the same data split for a fair validation comparison.
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


# Purpose: Centralizes filesystem paths so dataset inputs, caches, figures, models, and metric tables are easy
# Purpose: to trace.
PROJECT_ROOT = resolve_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "cnn"
CACHE_DIR = OUTPUT_DIR / "cache"
MANIFESTS_DIR = OUTPUT_DIR / "manifests"
CONFIGS_DIR = OUTPUT_DIR / "configs"
TABLES_DIR = OUTPUT_DIR / "tables"
HISTORIES_DIR = OUTPUT_DIR / "histories"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"
PILOT_LEGACY_DIR = OUTPUT_DIR / "pilot_legacy"
# Purpose: Creates each output directory before later cells try to save tables, figures, or models.
for directory in [OUTPUT_DIR, CACHE_DIR, MANIFESTS_DIR, CONFIGS_DIR, TABLES_DIR, HISTORIES_DIR, METRICS_DIR, FIGURES_DIR, MODELS_DIR, PILOT_LEGACY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def require_file(path):
    # Purpose: Stops the notebook early with a clear message when a required upstream artifact is missing.
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")
    return path


def load_json(path):
    # Purpose: Keeps the load_json helper isolated so later notebook cells can call it consistently.
    with open(require_file(path), "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(payload, path):
    # Purpose: Keeps the save_json helper isolated so later notebook cells can call it consistently.
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)




def normalise_handoff_config(config):
    # Purpose: Keeps rebuilt JSON handoffs compatible with the CNN builder.
    normalised = dict(config)
    for key in ["conv_filters", "kernel_size"]:
        value = normalised.get(key, [])
        if isinstance(value, str):
            value = json.loads(value)
        normalised[key] = [int(item) for item in value]
    normalised["dense_units"] = int(normalised["dense_units"])
    normalised["activation"] = str(normalised.get("activation", "relu"))
    normalised["optimizer"] = str(normalised.get("optimizer", "adam"))
    normalised["dropout"] = float(normalised["dropout"])
    normalised["learning_rate"] = float(normalised["learning_rate"])
    normalised["batch_size"] = int(normalised["batch_size"])
    normalised["loss"] = normalised.get("loss", "binary_crossentropy")
    normalised["threshold"] = float(normalised.get("threshold", 0.5))
    return normalised

def first_existing_results_path(candidate_paths):
    # Purpose: Supports the normal result filename and any documented alias without changing
    # where the notebook saves new results. The first existing CSV is used for rebuilding.
    paths = [Path(path) for path in candidate_paths]
    for path in paths:
        if path.exists():
            return path
    return paths[0]


def rebuild_selected_config_from_results(results_path, selected_path, selected_by_stage):
    # Purpose: Rebuilds a missing selected_config JSON from an existing validation-results
    # CSV. This avoids rerunning training just to recreate the JSON handoff file.
    results_path = Path(results_path)
    selected_path = Path(selected_path)
    if not results_path.exists():
        print("Cannot rebuild selected config because result CSV is missing:", results_path)
        return None

    results_df = pd.read_csv(results_path)
    required_columns = {"config_json", "validation_macro_f1", "best_validation_loss"}
    missing_columns = required_columns - set(results_df.columns)
    if missing_columns:
        raise RuntimeError(f"Cannot rebuild {selected_path.name}; missing columns in {results_path.name}: {sorted(missing_columns)}")
    if results_df.empty:
        raise RuntimeError(f"Cannot rebuild {selected_path.name}; result CSV is empty: {results_path}")

    best_row = results_df.sort_values(["validation_macro_f1", "best_validation_loss"], ascending=[False, True]).iloc[0]
    selected = normalise_handoff_config(json.loads(best_row["config_json"]))
    selected.update({
        "selected_by_stage": selected_by_stage,
        "selection_metric": "validation_macro_f1",
        "validation_macro_f1": float(best_row["validation_macro_f1"]),
        "best_validation_loss": float(best_row["best_validation_loss"]),
        "rebuilt_from_results_csv": str(results_path),
    })
    save_json(selected, selected_path)
    print("Rebuilt selected config from existing results:", selected_path)
    return selected


def ensure_selected_config_from_results(selected_path, results_path, selected_by_stage):
    # Purpose: Repairs a missing previous-stage config before this notebook tries to load it.
    # If neither the JSON nor the CSV exists, the later require_file/load_json call will still
    # stop with a clear missing-artifact error.
    selected_path = Path(selected_path)
    results_path = Path(results_path)
    if selected_path.exists():
        return load_json(selected_path)
    if results_path.exists():
        return rebuild_selected_config_from_results(results_path, selected_path, selected_by_stage)
    print("Selected config missing:", selected_path)
    print("Result CSV also missing, so config cannot be rebuilt:", results_path)
    return None


def load_class_weights():
    # Purpose: Keeps the load_class_weights helper isolated so later notebook cells can call it consistently.
    payload = load_json(CONFIGS_DIR / "class_weights.json")
    weights = payload.get("class_weights", payload)
    return {int(label): float(weight) for label, weight in weights.items()}


for required in [
    CACHE_DIR / "X_train.npy",
    CACHE_DIR / "y_train.npy",
    CACHE_DIR / "train_metadata.csv",
    CACHE_DIR / "X_validation.npy",
    CACHE_DIR / "y_validation.npy",
    CACHE_DIR / "validation_metadata.csv",
    CACHE_DIR / "feature_config.json",
    CONFIGS_DIR / "class_weights.json",
]:
    require_file(required)

X_train = np.load(CACHE_DIR / "X_train.npy", mmap_mode="r")[..., np.newaxis]
y_train = np.load(CACHE_DIR / "y_train.npy")
train_metadata = pd.read_csv(CACHE_DIR / "train_metadata.csv")
X_validation = np.load(CACHE_DIR / "X_validation.npy", mmap_mode="r")[..., np.newaxis]
y_validation = np.load(CACHE_DIR / "y_validation.npy")
validation_metadata = pd.read_csv(CACHE_DIR / "validation_metadata.csv")
feature_config = load_json(CACHE_DIR / "feature_config.json")
CLASS_WEIGHTS = load_class_weights()

if len(X_train) != len(y_train) or len(X_train) != len(train_metadata):
    raise RuntimeError("Training cache arrays, labels and metadata are not row-aligned.")
if len(X_validation) != len(y_validation) or len(X_validation) != len(validation_metadata):
    raise RuntimeError("Validation cache arrays, labels and metadata are not row-aligned.")

print("Loaded CNN cache:", CACHE_DIR)
print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)

# Purpose: Rebuild the previous selected config from its result CSV if the JSON handoff file is missing.
ensure_selected_config_from_results(CONFIGS_DIR / "03_selected_config.json", TABLES_DIR / "03_dense_units_results.csv", "03_dense_units")
base_config = load_json(CONFIGS_DIR / "03_selected_config.json")
print("Loaded previous config:", "03_selected_config.json")


Loaded CNN cache: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/cnn/cache
X_train shape: (9060, 40, 502, 1)
X_validation shape: (1942, 40, 502, 1)
Loaded previous config: 03_selected_config.json


## 3. Keras Utilities


In [4]:
# Purpose: Shared CNN utilities for validation-only model selection.
def normalise_config(config):
    # Purpose: Converts JSON/CSV values into the types expected by the CNN builder.
    normalised = dict(config)
    normalised["conv_filters"] = [int(value) for value in normalised["conv_filters"]]
    normalised["kernel_size"] = [int(value) for value in normalised["kernel_size"]]
    normalised["dense_units"] = int(normalised["dense_units"])
    normalised["activation"] = str(normalised.get("activation", "relu"))
    normalised["optimizer"] = str(normalised.get("optimizer", "adam"))
    normalised["dropout"] = float(normalised["dropout"])
    normalised["learning_rate"] = float(normalised["learning_rate"])
    normalised["batch_size"] = int(normalised["batch_size"])
    normalised["loss"] = normalised.get("loss", "binary_crossentropy")
    normalised["threshold"] = float(normalised.get("threshold", THRESHOLD))
    return normalised


def config_json(config):
    # Purpose: Creates a stable representation for deduplication and deterministic seeds.
    return json.dumps(normalise_config(config), sort_keys=True)


def config_key(config):
    return hashlib.sha256(config_json(config).encode("utf-8")).hexdigest()


def seed_from_config(config, base_seed=RANDOM_STATE):
    digest = hashlib.sha256(f"{base_seed}:{config_json(config)}".encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % (2**31 - 1)


def set_global_seed(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    tf.keras.utils.set_random_seed(int(seed))
    os.environ["PYTHONHASHSEED"] = str(int(seed))


def make_optimizer(config):
    name = str(config["optimizer"]).lower()
    learning_rate = float(config["learning_rate"])
    if name == "adam":
        return tf.keras.optimizers.Adam(learning_rate=learning_rate)
    if name == "rmsprop":
        return tf.keras.optimizers.RMSprop(learning_rate=learning_rate)
    if name in ["sgd", "sgd_momentum"]:
        return tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    raise ValueError(f"Unsupported optimizer: {config['optimizer']}")


def build_cnn_model(config, input_shape):
    # Purpose: Learns local patterns from the coefficient-by-time MFCC map.
    config = normalise_config(config)
    model = Sequential(name="mfcc_cnn")
    model.add(Input(shape=tuple(input_shape)))
    for filters in config["conv_filters"]:
        model.add(
            Conv2D(
                filters=filters,
                kernel_size=tuple(config["kernel_size"]),
                padding="same",
                strides=(1, 1),
                activation=config["activation"],
            )
        )
        model.add(MaxPooling2D(pool_size=(2, 2)))
        if config["dropout"] > 0:
            model.add(Dropout(config["dropout"]))
    # Global pooling controls parameter growth when the number of MFCC time frames changes.
    model.add(GlobalAveragePooling2D())
    model.add(Dense(config["dense_units"], activation=config["activation"]))
    if config["dropout"] > 0:
        model.add(Dropout(config["dropout"]))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(loss=config["loss"], optimizer=make_optimizer(config), metrics=["accuracy"])
    return model


def train_and_evaluate_config(config, run_seed, run_name, verbose=0):
    # Purpose: Uses training data for fitting and validation data for selection; never test data.
    config = normalise_config(config)
    tf.keras.backend.clear_session()
    set_global_seed(run_seed)
    model = build_cnn_model(config, X_train.shape[1:])
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOP_PATIENCE,
            min_delta=EARLY_STOP_MIN_DELTA,
            restore_best_weights=True,
        )
    ]
    start = time.perf_counter()
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=MAX_EPOCHS,
        batch_size=config["batch_size"],
        class_weight=CLASS_WEIGHTS,
        callbacks=callbacks,
        verbose=verbose,
    )
    runtime = time.perf_counter() - start
    probability = model.predict(X_validation, batch_size=config["batch_size"], verbose=0).ravel()
    pred = (probability >= config["threshold"]).astype(int)
    row = {
        "configuration": config,
        "config_json": config_json(config),
        "config_key": config_key(config),
        "seed": int(run_seed),
        "validation_macro_f1": float(f1_score(y_validation, pred, average="macro", zero_division=0)),
        "validation_binary_f1_synthetic": float(f1_score(y_validation, pred, pos_label=1, zero_division=0)),
        "validation_accuracy": float(accuracy_score(y_validation, pred)),
        "validation_precision_synthetic": float(precision_score(y_validation, pred, pos_label=1, zero_division=0)),
        "validation_recall_synthetic": float(recall_score(y_validation, pred, pos_label=1, zero_division=0)),
        "best_validation_loss": float(np.min(history.history["val_loss"])),
        "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
        "epochs_trained": int(len(history.history["loss"])),
        "runtime_seconds": float(runtime),
        "run_name": run_name,
    }
    history_df = pd.DataFrame(history.history)
    history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
    return row, history_df


def flat_result(row, extra=None):
    # Purpose: Makes each CNN configuration readable in CSV result tables.
    extra = extra or {}
    config = normalise_config(row["configuration"])
    flat = dict(extra)
    flat.update({
        "conv_filters": json.dumps(config["conv_filters"]),
        "kernel_size": json.dumps(config["kernel_size"]),
        "dense_units": config["dense_units"],
        "activation": config["activation"],
        "optimizer": config["optimizer"],
        "dropout": config["dropout"],
        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "loss": config["loss"],
        "threshold": config["threshold"],
        "config_json": row["config_json"],
        "config_key": row["config_key"],
        "seed": row["seed"],
        "validation_macro_f1": row["validation_macro_f1"],
        "validation_binary_f1_synthetic": row["validation_binary_f1_synthetic"],
        "validation_accuracy": row["validation_accuracy"],
        "validation_precision_synthetic": row["validation_precision_synthetic"],
        "validation_recall_synthetic": row["validation_recall_synthetic"],
        "best_validation_loss": row["best_validation_loss"],
        "best_epoch": row["best_epoch"],
        "epochs_trained": row["epochs_trained"],
        "runtime_seconds": row["runtime_seconds"],
    })
    return flat


def select_best(rows):
    return max(rows, key=lambda row: (row["validation_macro_f1"], -row["best_validation_loss"]))


def _controlled_candidate_label(value):
    # Purpose: Keeps list-valued candidates readable and stable in CSV outputs.
    if isinstance(value, (list, tuple, dict)):
        return json.dumps(value, sort_keys=True)
    return value


def run_controlled_trial(stage_name, varied_key, candidate_configs, results_path, histories_path, selected_path):
    # Purpose: Runs one controlled validation trial and writes the next-stage JSON handoff.
    if not candidate_configs:
        raise ValueError(f"No candidate configurations were provided for {stage_name}.")

    rows = []
    histories = {}
    total = len(candidate_configs)

    for index, raw_config in enumerate(candidate_configs, start=1):
        config = normalise_config(raw_config)
        candidate = _controlled_candidate_label(config[varied_key])
        run_name = f"{stage_name}_{index:02d}"
        print(f"Running {run_name} ({index}/{total}) with {varied_key}={candidate}")

        row, history_df = train_and_evaluate_config(
            config,
            run_seed=seed_from_config(config),
            run_name=run_name,
            verbose=2,
        )
        rows.append(row)
        histories[run_name] = history_df.to_dict(orient="records")

        partial_results = pd.DataFrame(
            [
                flat_result(
                    trial_row,
                    {
                        "stage": stage_name,
                        "candidate": _controlled_candidate_label(trial_row["configuration"][varied_key]),
                    },
                )
                for trial_row in rows
            ]
        )
        partial_results.to_csv(results_path, index=False)
        save_json(histories, histories_path)

    results_df = pd.DataFrame(
        [
            flat_result(
                trial_row,
                {
                    "stage": stage_name,
                    "candidate": _controlled_candidate_label(trial_row["configuration"][varied_key]),
                },
            )
            for trial_row in rows
        ]
    ).sort_values(["validation_macro_f1", "best_validation_loss"], ascending=[False, True])

    best = select_best(rows)
    selected = normalise_config(best["configuration"])
    selected.update(
        {
            "selected_by_stage": stage_name,
            "selection_metric": "validation_macro_f1",
            "varied_key": varied_key,
            "selected_candidate": _controlled_candidate_label(selected[varied_key]),
            "validation_macro_f1": float(best["validation_macro_f1"]),
            "best_validation_loss": float(best["best_validation_loss"]),
            "config_key": best["config_key"],
            "seed": int(best["seed"]),
        }
    )
    save_json(selected, selected_path)

    display(results_df)
    print("Saved results:", results_path)
    print("Saved histories:", histories_path)
    print("Saved selected config:", selected_path)
    return results_df


## 4. Candidate Configurations


In [5]:
# Purpose: 4. Candidate Configurations.
VARIED_KEY = "dropout"
candidate_values = [0.2, 0.3, 0.4, 0.5]
candidate_configs = []
# Purpose: Runs the controlled trial once for each candidate value of the tested hyperparameter.
for value in candidate_values:
    config = dict(base_config)
    config[VARIED_KEY] = value
    candidate_configs.append(normalise_config(config))
display(pd.DataFrame(candidate_configs))


,conv_filters,kernel_size,dense_units,activation,optimizer,dropout,learning_rate,batch_size,loss,threshold,selected_by_stage,selection_metric,varied_key,selected_candidate,validation_macro_f1,best_validation_loss,config_key,seed
0,"[32, 64]","[5, 3]",128,relu,adam,0.2,0.001,128,binary_crossentropy,0.5,03_dense_units,validation_macro_f1,dense_units,128,0.960805,0.097529,820ed354495ca5ca5c327cc6e195dca7e6001eae220021...,1090814782
1,"[32, 64]","[5, 3]",128,relu,adam,0.3,0.001,128,binary_crossentropy,0.5,03_dense_units,validation_macro_f1,dense_units,128,0.960805,0.097529,820ed354495ca5ca5c327cc6e195dca7e6001eae220021...,1090814782
2,"[32, 64]","[5, 3]",128,relu,adam,0.4,0.001,128,binary_crossentropy,0.5,03_dense_units,validation_macro_f1,dense_units,128,0.960805,0.097529,820ed354495ca5ca5c327cc6e195dca7e6001eae220021...,1090814782
3,"[32, 64]","[5, 3]",128,relu,adam,0.5,0.001,128,binary_crossentropy,0.5,03_dense_units,validation_macro_f1,dense_units,128,0.960805,0.097529,820ed354495ca5ca5c327cc6e195dca7e6001eae220021...,1090814782


## 5. Run or Load Results


In [6]:
# Purpose: Runs the trial when enabled, otherwise repairs or displays the selected config
# handoff from existing validation results without retraining.
RUN_DROPOUT_TRIAL = True
results_path = TABLES_DIR / "04_dropout_results.csv"
histories_path = HISTORIES_DIR / "04_dropout_histories.json"
selected_path = CONFIGS_DIR / "04_selected_config.json"
rebuild_results_path = first_existing_results_path([TABLES_DIR / "04_dropout_results.csv"])

if RUN_DROPOUT_TRIAL:
    run_controlled_trial("04_dropout", VARIED_KEY, candidate_configs, results_path, histories_path, selected_path)
elif selected_path.exists():
    selected_config = load_json(selected_path)
    print("Loaded selected config:", selected_path)
    if rebuild_results_path.exists():
        display(pd.read_csv(rebuild_results_path).sort_values(["validation_macro_f1", "best_validation_loss"], ascending=[False, True]))
    else:
        display(pd.DataFrame([selected_config]))
elif rebuild_results_path.exists():
    rebuild_selected_config_from_results(rebuild_results_path, selected_path, "04_dropout")
    display(pd.read_csv(rebuild_results_path).sort_values(["validation_macro_f1", "best_validation_loss"], ascending=[False, True]))
else:
    print("RUN_DROPOUT_TRIAL is False. [RESULT TO BE INSERTED AFTER FINAL RUN]")
    print("Missing selected config:", selected_path)
    print("Missing result CSV:", rebuild_results_path)


Running 04_dropout_01 (1/4) with dropout=0.2
Epoch 1/50
71/71 - 28s - 392ms/step - accuracy: 0.5663 - loss: 0.6721 - val_accuracy: 0.6256 - val_loss: 0.6356
Epoch 2/50
71/71 - 1s - 15ms/step - accuracy: 0.6389 - loss: 0.6183 - val_accuracy: 0.7049 - val_loss: 0.5463
Epoch 3/50
71/71 - 1s - 15ms/step - accuracy: 0.7109 - loss: 0.5474 - val_accuracy: 0.7147 - val_loss: 0.5740
Epoch 4/50
71/71 - 1s - 15ms/step - accuracy: 0.7589 - loss: 0.4904 - val_accuracy: 0.7528 - val_loss: 0.5182
Epoch 5/50
71/71 - 1s - 15ms/step - accuracy: 0.7822 - loss: 0.4499 - val_accuracy: 0.7920 - val_loss: 0.4587
Epoch 6/50
71/71 - 1s - 15ms/step - accuracy: 0.8010 - loss: 0.4145 - val_accuracy: 0.7899 - val_loss: 0.4533
Epoch 7/50
71/71 - 1s - 15ms/step - accuracy: 0.8224 - loss: 0.3763 - val_accuracy: 0.8074 - val_loss: 0.4187
Epoch 8/50
71/71 - 1s - 15ms/step - accuracy: 0.8415 - loss: 0.3482 - val_accuracy: 0.8162 - val_loss: 0.3987
Epoch 9/50
71/71 - 1s - 15ms/step - accuracy: 0.8535 - loss: 0.3150 - val

,stage,candidate,conv_filters,kernel_size,dense_units,activation,optimizer,dropout,learning_rate,batch_size,...,seed,validation_macro_f1,validation_binary_f1_synthetic,validation_accuracy,validation_precision_synthetic,validation_recall_synthetic,best_validation_loss,best_epoch,epochs_trained,runtime_seconds
3,04_dropout,0.5,"[32, 64]","[5, 3]",128,relu,adam,0.5,0.001,128,...,1151998753,0.942204,0.962054,0.949022,0.991311,0.934475,0.136563,45,50,62.471324
1,04_dropout,0.3,"[32, 64]","[5, 3]",128,relu,adam,0.3,0.001,128,...,94885234,0.938483,0.959136,0.945417,0.994404,0.926284,0.146031,37,44,55.936409
2,04_dropout,0.4,"[32, 64]","[5, 3]",128,relu,adam,0.4,0.001,128,...,1240344849,0.933542,0.955478,0.940783,0.995161,0.918838,0.145092,48,50,62.396139
0,04_dropout,0.2,"[32, 64]","[5, 3]",128,relu,adam,0.2,0.001,128,...,734875399,0.925509,0.950019,0.933574,0.990307,0.912882,0.163499,24,31,83.747285


Saved results: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/cnn/tables/04_dropout_results.csv
Saved histories: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/cnn/histories/04_dropout_histories.json
Saved selected config: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/cnn/configs/04_selected_config.json
